In [2]:
import pyodbc
import pandas as pd

# --- 1. 配置区 ---
db_path = r"D:\数据\资产配置\处理后数据\Data-Fund.accdb"
target_code = ("000007")
start_date_str = "2023-03-01"
end_date_str = "2025-12-31"

static_table = '基金主要信息'
timeseries_tables = [
    '资产配置2303-2512',
    '股票配置2303-2512',
    '按行业股票配置表2303-2512',
    '债券配置2303-2512'
]

conn_str = f"DRIVER={{Microsoft Access Driver (*.mdb, *.accdb)}};DBQ={db_path};"

try:
    conn = pyodbc.connect(conn_str)

    # --- 2. 提取静态信息 ---
    profile_query = f"SELECT * FROM [{static_table}] WHERE [MasterFundCode] = '{target_code}'"
    profile_df = pd.read_sql(profile_query, conn)

    # --- 3. 提取动态时间序列 ---
    ts_dfs = []
    for table in timeseries_tables:
        # 获取列名
        cols_query = f"SELECT TOP 1 * FROM [{table}]"
        actual_cols = pd.read_sql(cols_query, conn).columns.tolist()
        date_col = next((c for c in actual_cols if c.upper() == 'ENDDATE'), 'EndDate')
        code_col = next((c for c in actual_cols if c.upper() == 'MASTERFUNDCODE'), 'MasterFundCode')
        query = f"SELECT * FROM [{table}] WHERE [{code_col}] = '{target_code}'"
        df = pd.read_sql(query, conn)

        if not df.empty:
            # 统一字段名
            df = df.rename(columns={code_col: 'MasterFundCode', date_col: 'EndDate'})

            # 强制转换 EndDate 为 Pandas 时间对象，以便进行可靠筛选
            df['EndDate'] = pd.to_datetime(df['EndDate'], errors='coerce')

            # 【内存筛选】：在 Python 中进行日期过滤，避免 Access SQL 的 Bug
            mask = (df['EndDate'] >= start_date_str) & (df['EndDate'] <= end_date_str)
            df = df.loc[mask].copy()

            if not df.empty:
                # 转换回字符串显示格式
                df['EndDate'] = df['EndDate'].dt.strftime('%Y-%m-%d')

                # 清洗代码格式
                df['MasterFundCode'] = df['MasterFundCode'].apply(lambda x: str(int(float(x))).zfill(6) if pd.notnull(x) else "")

                # 处理明细序号
                df['Seq'] = df.groupby(['MasterFundCode', 'EndDate']).cumcount() + 1
                df = df.set_index(['MasterFundCode', 'EndDate', 'Seq'])

                # 加前缀
                df.columns = [f"{table}_{col}" for col in df.columns]
                ts_dfs.append(df)
                print(f"  - 成功提取并过滤数据: {table} (含 2025 数据: {'是' if '2025' in str(df.index) else '否'})")

    # --- 4. 生成报告 ---
    output_file = f"Fund_Full_Report_{target_code}.md"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"# 基金分析报告: {target_code} (2023-2025)\n\n")
        f.write("## [附注: 基金基础信息]\n")
        f.write(profile_df.to_markdown(index=False) if not profile_df.empty else "无数据")

        f.write("\n\n---\n\n")
        f.write("## [历史配置明细数据]\n")

        if ts_dfs:
            final_ts_df = pd.concat(ts_dfs, axis=1, join='outer').reset_index()
            # 确保排序正确
            final_ts_df = final_ts_df.sort_values(by=['EndDate', 'Seq'], ascending=[False, True])
            f.write(final_ts_df.fillna("-").to_markdown(index=False))
        else:
            f.write("未能提取到有效的时间序列数据，请检查数据库连接。")

    print(f"\n新报告已生成: {output_file}")
    conn.close()

except Exception as e:
    print(f"执行失败: {e}")

C:\Users\qyw\AppData\Local\Temp\ipykernel_13940\1066014465.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  profile_df = pd.read_sql(profile_query, conn)
C:\Users\qyw\AppData\Local\Temp\ipykernel_13940\1066014465.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  actual_cols = pd.read_sql(cols_query, conn).columns.tolist()
C:\Users\qyw\AppData\Local\Temp\ipykernel_13940\1066014465.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
C:\Users\qyw\AppData\Local\Temp\ipykernel_13940\10660144


新报告已生成: Fund_Full_Report_000007.md


C:\Users\qyw\AppData\Local\Temp\ipykernel_13940\1066014465.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  actual_cols = pd.read_sql(cols_query, conn).columns.tolist()
C:\Users\qyw\AppData\Local\Temp\ipykernel_13940\1066014465.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
